# Seto — Kaggle Training Pipeline

Wraps `scripts/prepare_data.py` and `scripts/train.py` via `torchrun`.

## Stages
1. **Prepare data** — download + tokenize + shard
2. **Pretrain** — next-token prediction
3. **Cooldown** — lower LR, same data
4. **SFT** — instruction following
5. **DPO** — preference optimization

## Setup
1. Upload `seto/` repo (with `scripts/`) as Kaggle dataset
2. Run cells in order

In [ ]:
!pip install -q tokenizers datasets

In [ ]:
import torch, os, sys, shutil, glob
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_mem/1e9:.1f}GB)")

In [ ]:
SETO_SRC = "/kaggle/input/seto-model"
WORKING = "/kaggle/working"
for directory in ["seto", "scripts"]:
    src = f"{SETO_SRC}/{directory}"
    dst = f"{WORKING}/{directory}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
os.chdir(WORKING)
sys.path.insert(0, WORKING)
print(f"seto/ contents: {os.listdir('seto/')}")
print(f"scripts/ contents: {os.listdir('scripts/')}")

In [ ]:
# ============================================================
# CONFIGURE THESE
# ============================================================
MODEL = "tiny"            # tiny | small | base
TOKENIZER_DIR = "seto-tokenizer"
DATA_DIR = "data/shards"
OUTPUT_DIR = "seto-output"
NUM_GPUS = torch.cuda.device_count()
BATCH_SIZE = 4
GRAD_ACCUM = 2
print(f"Model: {MODEL} | GPUs: {NUM_GPUS} | BS: {BATCH_SIZE}x{GRAD_ACCUM}x{NUM_GPUS}={BATCH_SIZE*GRAD_ACCUM*NUM_GPUS}")

In [ ]:
def latest_checkpoint(stage):
    """Find latest zip checkpoint for a stage."""
    ckpt_dir = f"{OUTPUT_DIR}/checkpoints_{stage}"
    files = sorted(glob.glob(f"{ckpt_dir}/seto_step_*.zip"),
                   key=os.path.getmtime)
    if not files:
        raise FileNotFoundError(f"No checkpoint for stage '{stage}' in {ckpt_dir}")
    return files[-1]

In [ ]:
# ============================================================
# STAGE 1: PREPARE DATA (tokenizer + shards)
# ============================================================
SKIP_PREPARE = os.path.exists(DATA_DIR) and any(f.endswith('.bin') for f in os.listdir(DATA_DIR))

if not SKIP_PREPARE:
    !python scripts/prepare_data.py \
        --output-dir data \
        --tokenizer-dir {TOKENIZER_DIR} \
        --max-samples-ru 100000 \
        --max-samples-wiki 20000
else:
    print(f"Data already exists at {DATA_DIR}")
    print(f"  Files: {os.listdir(DATA_DIR)[:10]}")

In [ ]:
# ============================================================
# STAGE 2: PRETRAIN
# ============================================================
PRETRAIN_STEPS = 50000

!torchrun --nproc_per_node={NUM_GPUS} scripts/train.py \
    --stage pretrain \
    --model-config {MODEL} \
    --data-dir {DATA_DIR} \
    --output-dir {OUTPUT_DIR} \
    --tokenizer {TOKENIZER_DIR} \
    --batch-size {BATCH_SIZE} \
    --grad-accum {GRAD_ACCUM} \
    --max-steps {PRETRAIN_STEPS}

In [ ]:
# ============================================================
# STAGE 3: COOLDOWN (optional, skip for short runs)
# ============================================================
SKIP_COOLDOWN = True

if not SKIP_COOLDOWN:
    PRETRAIN_CKPT = latest_checkpoint("pretrain")
    print(f"Loading from: {PRETRAIN_CKPT}")
    !torchrun --nproc_per_node={NUM_GPUS} scripts/train.py \
        --stage cooldown \
        --model-config {MODEL} \
        --data-dir {DATA_DIR} \
        --output-dir {OUTPUT_DIR} \
        --tokenizer {TOKENIZER_DIR} \
        --batch-size {BATCH_SIZE} \
        --grad-accum {GRAD_ACCUM} \
        --max-steps 5000 \
        --init-from {PRETRAIN_CKPT}
else:
    print("Cooldown skipped")

In [ ]:
# ============================================================
# STAGE 4: SFT
# ============================================================
# Needs conversation data: {"messages": [{"role": ..., "content": ...}, ...]}
# Sources: SmolTalk, UltraChat, saiga_scored

SFT_DATA = None  # Set to path with .jsonl/.json files

if SFT_DATA:
    # Use latest pretrain checkpoint (or cooldown if it ran)
    try:
        INIT_FROM = latest_checkpoint("cooldown")
    except FileNotFoundError:
        INIT_FROM = latest_checkpoint("pretrain")
    print(f"Loading from: {INIT_FROM}")
    !torchrun --nproc_per_node={NUM_GPUS} scripts/train.py \
        --stage sft \
        --model-config {MODEL} \
        --data-dir {SFT_DATA} \
        --output-dir {OUTPUT_DIR} \
        --tokenizer {TOKENIZER_DIR} \
        --batch-size {BATCH_SIZE} \
        --grad-accum {GRAD_ACCUM} \
        --max-steps 5000 \
        --init-from {INIT_FROM}
else:
    print("No SFT data. Set SFT_DATA and re-run.")

In [ ]:
# ============================================================
# STAGE 5: DPO
# ============================================================
# Needs: {"prompt": ..., "chosen": ..., "rejected": ...}

DPO_DATA = None

if DPO_DATA:
    SFT_CKPT = latest_checkpoint("sft")
    print(f"Loading from: {SFT_CKPT}")
    !torchrun --nproc_per_node={NUM_GPUS} scripts/train.py \
        --stage dpo \
        --model-config {MODEL} \
        --data-dir {DPO_DATA} \
        --output-dir {OUTPUT_DIR} \
        --tokenizer {TOKENIZER_DIR} \
        --batch-size {BATCH_SIZE} \
        --grad-accum {GRAD_ACCUM} \
        --max-steps 2000 \
        --init-from {SFT_CKPT} \
        --dpo-ref-model {SFT_CKPT}
else:
    print("No DPO data. Set DPO_DATA and re-run.")

In [ ]:
# ============================================================
# LIST CHECKPOINTS
# ============================================================
for stage in ["pretrain", "cooldown", "sft", "dpo"]:
    ckpt_dir = f"{OUTPUT_DIR}/checkpoints_{stage}"
    zips = sorted(glob.glob(f"{ckpt_dir}/seto_step_*.zip"),
                  key=os.path.getmtime)
    if zips:
        print(f"\n{stage} ({len(zips)} checkpoints):")
        for z in zips:
            print(f"  {os.path.basename(z)} ({os.path.getsize(z)/1e6:.1f} MB)")

In [ ]:
# ============================================================
# QUICK INFERENCE TEST (loads most advanced stage)
# ============================================================
import json
from seto import SetoLM, SetoTokenizer
from seto.config import MODEL_TINY, MODEL_SMALL, MODEL_BASE

model_map = {"tiny": MODEL_TINY, "small": MODEL_SMALL, "base": MODEL_BASE}
cfg = model_map[MODEL]
tokenizer = SetoTokenizer.from_pretrained(TOKENIZER_DIR)

# Try stages in reverse order: most advanced first
final_dir = None
for stage in ["dpo", "sft", "cooldown", "pretrain"]:
    candidate = f"{OUTPUT_DIR}/final_{stage}"
    if os.path.exists(f"{candidate}/model.pt"):
        final_dir = candidate
        break

if final_dir:
    model = SetoLM(cfg)
    model.load_state_dict(torch.load(f"{final_dir}/model.pt", weights_only=True))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()

    prompt = "<|system|>\nYou are Seto.\n<|user|>\nWhat is 2+2?\n<|assistant|>\n"
    ids = torch.tensor([tokenizer.encode(prompt, add_bos=True, add_eos=False)], device=device)
    with torch.no_grad():
        for _ in range(100):
            logits, _ = model(ids)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            ids = torch.cat([ids, next_token], dim=-1)
    print(tokenizer.decode(ids[0].tolist(), skip_special_tokens=True))
else:
    print("No trained model found yet")